# NSF Funding Analysis (2021–2025)
**DS 5500 — Chenjie Gu**

This notebook analyzes NSF award data from 2021 to 2025, covering:
1. Funding trends over time by program/CFDA category
2. Top-funded institutions and states per year
3. Program Officer award counts
4. Topic modeling on award abstracts using LDA and BERTopic

> **Run cells in order top-to-bottom.** Each section depends on variables defined above it.  
> All plots are saved as PNG files and then compiled into a single PDF at the end.

## 1. Setup & Installations

In [1]:
import sys
!{sys.executable} -m pip install -q "urllib3<2" seaborn plotly pandas numpy matplotlib openpyxl geopandas
!{sys.executable} -m pip install -q nltk gensim pyLDAvis
!{sys.executable} -m pip install -q bertopic sentence-transformers umap-learn hdbscan scikit-learn
!{sys.executable} -m pip install -q pypdf Pillow kaleido
!pip install statsmodels
print("All packages installed.")

All packages installed.


In [2]:
import os, re, ast, warnings, logging
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — avoids display errors
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet',   quiet=True)

import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import pyLDAvis
import pyLDAvis.gensim_models

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
logging.getLogger('gensim').setLevel(logging.ERROR)
logging.getLogger('bertopic').setLevel(logging.ERROR)


plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

# ── Output folder for saved plots ─────────────────────────────────────────────
PLOT_DIR = 'plots'
os.makedirs(PLOT_DIR, exist_ok=True)
SAVED_PLOTS = []   # running list — populated by save_fig() below

def save_fig(fig_or_name, name):
    """Save a matplotlib Figure or a Plotly figure to PLOT_DIR and register it."""
    path = os.path.join(PLOT_DIR, f'{name}.png')
    if hasattr(fig_or_name, 'write_image'):           # Plotly
        fig_or_name.write_image(path, width=1200, height=600, scale=2)
    elif hasattr(fig_or_name, 'savefig'):              # Matplotlib Figure
        fig_or_name.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig_or_name)
    SAVED_PLOTS.append(path)
    print(f'  Saved → {path}')

print("All imports successful.")
print(f"Plots will be saved to: {os.path.abspath(PLOT_DIR)}")

All imports successful.
Plots will be saved to: /Users/_fin.fish_/Desktop/NEU/DS5500/plots


## 2. Load & Inspect Data

In [3]:
df = pd.read_excel('NSF_21_25_SG.xlsx', engine='openpyxl')
print(f'Loaded {len(df):,} rows')
print(df.dtypes.head(20))

Loaded 6,477 rows
Unnamed: 0.2                        int64
Unnamed: 0.1                        int64
Unnamed: 0                          int64
awd_id                              int64
agcy_id                            object
tran_type                          object
awd_istr_txt                       object
awd_titl_txt                       object
cfda_num                           object
org_code                            int64
po_phone                          float64
po_email                           object
po_sign_block_name                 object
awd_eff_date               datetime64[ns]
awd_exp_date               datetime64[ns]
tot_intn_awd_amt                    int64
awd_amount                          int64
awd_min_amd_letter_date    datetime64[ns]
awd_max_amd_letter_date    datetime64[ns]
abstract                           object
dtype: object


In [4]:
# ── Data Cleaning ─────────────────────────────────────────────────────────────

df['awd_eff_date']   = pd.to_datetime(df['awd_eff_date'],  errors='coerce')
df['awd_exp_date']   = pd.to_datetime(df['awd_exp_date'],  errors='coerce')
df['year']           = df['awd_eff_year'].astype(int)
df['amount']         = pd.to_numeric(df['awd_amount'],      errors='coerce')
df['duration_years'] = ((df['awd_exp_date'] - df['awd_eff_date']).dt.days / 365.25).round(2)
df['institution']    = df['inst.inst_name']
df['state']          = df['inst.inst_state_code']
df['country']        = df['inst.inst_country_name']

# ── Extract PI name from nested JSON-like string ──────────────────────────────
def extract_pi_name(pi_str):
    try:
        cleaned = str(pi_str).replace('None', '"None"')
        records = ast.literal_eval(cleaned)
        for r in records:
            if r.get('pi_role') == 'Principal Investigator':
                return r.get('pi_full_name', '')
        return records[0].get('pi_full_name', '') if records else ''
    except:
        return ''

# ── Extract max obligation amount ────────────────────────────────────────────
def extract_oblg(oblg_str):
    try:
        records = ast.literal_eval(str(oblg_str))
        if records:
            return max(r.get('fund_oblg_amt', 0) for r in records)
    except:
        pass
    return np.nan

df['pi_name']      = df['pi'].apply(extract_pi_name)
df['oblg_amt']     = df['oblg_fy'].apply(extract_oblg)
df['cfda_primary'] = df['cfda_num'].astype(str).str.split(',').str[0].str.strip()

# ── Filter to 2021-2025 ───────────────────────────────────────────────────────
df = df[df['year'].between(2021, 2025)].copy()

print(f'Years: {df.year.min()} – {df.year.max()}')
print(f'Filtered to {len(df):,} awards (2021–2025)')
print(f'Missing amounts: {df.amount.isna().sum()}')
print(df.year.value_counts().sort_index())

Years: 2021 – 2025
Filtered to 6,398 awards (2021–2025)
Missing amounts: 0
year
2021    1278
2022    1441
2023    1462
2024    1194
2025    1023
Name: count, dtype: int64


## 3. Funding Trends Over Time

In [5]:
# ── Total NSF funding per year ────────────────────────────────────────────────
yearly = df.groupby('year')['amount'].sum().reset_index()
yearly.columns = ['Year', 'Total Funding ($)']

fig = px.bar(
    yearly, x='Year', y='Total Funding ($)',
    title='Total NSF Funding per Year (2021–2025)',
    text_auto='.2s'
)
fig.update_traces(marker_color='steelblue')
fig.update_layout(showlegend=False)
save_fig(fig, '01_total_funding_per_year')
fig.show()

  Saved → plots/01_total_funding_per_year.png


In [6]:
# ── Funding by CFDA category ──────────────────────────────────────────────────
df_cfda = df.copy()
df_cfda['cfda_split'] = df_cfda['cfda_num'].astype(str).str.split(',')
df_cfda = df_cfda.explode('cfda_split')
df_cfda['cfda_split'] = df_cfda['cfda_split'].str.strip()
df_cfda['cfda_split'] = df_cfda['cfda_split'].replace('47.070', '47.07')

cfda_names = {
    '47.07':  'Computer & Info Science',
    '47.049': 'Math & Physical Sciences',
    '47.050': 'Geosciences',
    '47.074': 'Biological Sciences',
    '47.075': 'Social Sciences',
    '47.076': 'Education',
    '47.079': 'STEM Education',
    '47.083': 'Office of Integrative Activities',
    '47.041': 'Engineering',
}
df_cfda['cfda_label'] = df_cfda['cfda_split'].map(cfda_names).fillna(df_cfda['cfda_split'])

cfda_year = df_cfda.groupby(['year', 'cfda_label'])['amount'].sum().reset_index()
top_cfda = (
    cfda_year.groupby('cfda_label')['amount']
    .sum().nlargest(6).index.tolist()
)

fig = px.line(
    cfda_year[cfda_year['cfda_label'] == 'Computer & Info Science'],
    x='year', y='amount', color='cfda_label',
    title='Computer & Info Science Funding Over Time',
    labels={'amount': 'Total Funding ($)', 'year': 'Year', 'cfda_label': 'CFDA Category'},
    markers=True
)
save_fig(fig, '02_cfda_funding_trends')
fig.show()

  Saved → plots/02_cfda_funding_trends.png


In [7]:
# ── Treemap: total funding by CFDA ───────────────────────────────────────────
cfda_total = cfda_year.groupby('cfda_label')['amount'].sum().reset_index()

fig = px.treemap(
    cfda_total, path=['cfda_label'], values='amount',
    title='Total NSF Funding by CFDA Category (2021–2025)',
    color='amount', color_continuous_scale='Blues',
    labels={'amount': 'Total Funding ($)'}
)
fig.update_traces(textinfo='label+value+percent root')
save_fig(fig, '04_cfda_treemap')
fig.show()

  Saved → plots/04_cfda_treemap.png


In [8]:
# ── CSE division funding per year ─────────────────────────────────────────────
print('Division value counts:')
print(df['div_abbr'].value_counts())

div_yearly = df.groupby(['year', 'div_abbr']).agg(
    total_funding=('amount', 'sum'),
    num_awards=('awd_id', 'count')
).reset_index()

div_names = {
    'CSE': 'Computer and Information Science and Engineering (CISE)',
    'OAC': 'Advanced Cyberinfrastructure (CISE/OAC)',
    'CNS': 'Computer and Network Systems (CISE/CNS)',
    'CCF': 'Computing and Communication Foundations (CISE/CCF)',
    'IIS': 'Information and Intelligent Systems (CISE/IIS)',
}
div_yearly['division'] = div_yearly['div_abbr'].map(div_names).fillna(div_yearly['div_abbr'])

fig1 = px.line(
    div_yearly, x='year', y='total_funding', color='division',
    markers=True,
    title='NSF CSE Funding by Division per Year',
    labels={'total_funding': 'Total Funding ($)', 'year': 'Year', 'division': 'Division'}
)
fig1.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
save_fig(fig1, '05_div_funding_per_year')
fig1.show()

fig2 = px.line(
    div_yearly, x='year', y='num_awards', color='division',
    markers=True,
    title='NSF CSE Award Count by Division per Year',
    labels={'num_awards': 'Number of Awards', 'year': 'Year', 'division': 'Division'}
)
fig2.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
save_fig(fig2, '06_div_awards_per_year')
fig2.show()

Division value counts:
div_abbr
CNS    2166
IIS    1601
CCF    1445
OAC    1186
Name: count, dtype: int64
  Saved → plots/05_div_funding_per_year.png


  Saved → plots/06_div_awards_per_year.png


In [9]:
# ── Division totals (all years combined) ─────────────────────────────────────
div_total = (
    df.groupby('div_abbr').agg(
        total_funding=('amount', 'sum'),
        num_awards=('awd_id', 'count')
    ).reset_index()
    .sort_values('total_funding', ascending=False)
)
div_names2 = {
    'OAC': 'Advanced Cyberinfrastructure (OAC)',
    'CNS': 'Computer and Network Systems (CNS)',
    'CCF': 'Computing and Communication Foundations (CCF)',
    'IIS': 'Information and Intelligent Systems (IIS)',
}
div_total['division'] = div_total['div_abbr'].map(div_names2).fillna(div_total['div_abbr'])

fig1 = px.bar(
    div_total, x='division', y='total_funding',
    title='NSF CSE Total Funding by Division (2021–2025)',
    labels={'total_funding': 'Total Funding ($)', 'division': 'Division'},
    height=500
)
fig1.update_traces(marker_color='steelblue')
fig1.update_layout(xaxis_tickangle=-15)
save_fig(fig1, '07_div_total_funding')
fig1.show()

fig2 = px.bar(
    div_total, x='division', y='num_awards',
    title='NSF CSE Total Awards by Division (2021–2025)',
    labels={'num_awards': 'Number of Awards', 'division': 'Division'},
    height=500
)
fig2.update_traces(marker_color='steelblue')
fig2.update_layout(xaxis_tickangle=-15)
save_fig(fig2, '08_div_total_awards')
fig2.show()

  Saved → plots/07_div_total_funding.png


  Saved → plots/08_div_total_awards.png


## 4. Institutional & Geographic Analysis

In [10]:
# ── Institution totals ────────────────────────────────────────────────────────
import numpy as np
import plotly.graph_objects as go

inst_total = (
    df.groupby('institution')['amount']
    .sum().reset_index()
    .sort_values('amount', ascending=False)
    .reset_index(drop=True)
)
inst_total = inst_total[inst_total['amount'] > 0].reset_index(drop=True)
inst_total['rank'] = np.arange(len(inst_total))

fig1 = px.bar(
    inst_total, x='institution', y='amount',
    title='Total NSF CSE Funding by Institution (2021–2025)',
    labels={'amount': 'Total Funding ($)', 'institution': 'Institution'},
    height=600
)
fig1.update_traces(marker_color='steelblue')

# analytically fit exponential through top, middle, and bottom points
n = len(inst_total)
x0, y0 = 0,        inst_total['amount'].iloc[0]       # top institution
x1, y1 = n // 2,   inst_total['amount'].iloc[n // 2]  # median institution
x2, y2 = n - 1,    inst_total['amount'].iloc[-1]       # bottom institution

# solve a*e^(-b*x) + c through the three anchor points
c = (y1**2 - y0*y2) / (2*y1 - y0 - y2)
a = (y0 - c)
b = -np.log((y1 - c) / a) / x1
trend_vals = a * np.exp(-b * inst_total['rank']) + c

fig1.add_trace(go.Scatter(
    x=inst_total['institution'], y=trend_vals,
    mode='lines', name='Trend', line=dict(color='red', width=2)
))

fig1.update_layout(xaxis_tickangle=-90, xaxis_tickfont_size=7)
save_fig(fig1, '10_inst_total_bar')
fig1.show()

fig2 = px.treemap(
    inst_total, path=['institution'], values='amount',
    title='NSF CSE Funding by Institution — Treemap (2021–2025)',
    color='amount', color_continuous_scale='Blues',
)
fig2.update_traces(textinfo='label+value')
save_fig(fig2, '11_inst_treemap')
fig2.show()

inst_counts = (
    df.groupby('institution').agg(
        total_funding=('amount', 'sum'),
        num_awards=('awd_id', 'count')
    ).reset_index()
    .sort_values('total_funding', ascending=False)
    .head(50)
)
fig3 = px.scatter(
    inst_counts, x='num_awards', y='total_funding',
    size='total_funding', hover_name='institution',
    text='institution',
    title='NSF CSE Institutions — Bubble Chart (Top 50, 2021–2025)',
    labels={'total_funding': 'Total Funding ($)', 'num_awards': 'Number of Awards'},
    color='total_funding', color_continuous_scale='Teal', height=600
)
fig3.update_traces(textposition='top center', textfont_size=7)
save_fig(fig3, '12_inst_bubble')
fig3.show()

  Saved → plots/10_inst_total_bar.png


  Saved → plots/11_inst_treemap.png


  Saved → plots/12_inst_bubble.png


In [11]:
# ── State-level funding maps ───────────────────────────────────────────────────
import geopandas as gpd

state_year = (
    df.groupby(['year', 'state'])['amount']
    .sum().reset_index()
)
print('Top states by total funding:')
print(state_year.groupby('state')['amount'].sum().nlargest(10))

gdf = gpd.read_file(
    'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/'
    'geojson/ne_110m_admin_1_states_provinces.geojson'
)
gdf = gdf[gdf['iso_a2'] == 'US'].copy()
gdf['state'] = gdf['postal'].str.strip()

vmin = state_year['amount'].min()
vmax = state_year['amount'].max()
years = sorted(state_year['year'].unique())

for yr in years:
    yr_data = state_year[state_year['year'] == yr]
    merged  = gdf.merge(yr_data, on='state', how='left')

    fig_map, ax = plt.subplots(1, 1, figsize=(14, 8))
    merged.plot(
        column='amount', ax=ax, cmap='YlOrRd',
        vmin=vmin, vmax=vmax,
        edgecolor='black', linewidth=0.3,
        legend=True,
        legend_kwds={
            'label': 'Total Funding ($)',
            'orientation': 'vertical',
            'shrink': 0.6,
            'format': lambda x, _: f'${x/1e6:.0f}M'
        },
        missing_kwds={'color': 'lightgrey', 'label': 'No Data'}
    )
    ax.set_title(f'NSF CSE Funding by State — {yr}', fontsize=16, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    save_fig(fig_map, f'13_state_map_{yr}')
    plt.show()

Top states by total funding:
state
CA    364532018
NY    214423772
TX    187578298
MA    176824420
IL    172946611
PA    152805549
VA    109501412
IN    100800849
NJ     92184105
MI     90980418
Name: amount, dtype: int64
  Saved → plots/13_state_map_2021.png
  Saved → plots/13_state_map_2022.png
  Saved → plots/13_state_map_2023.png
  Saved → plots/13_state_map_2024.png
  Saved → plots/13_state_map_2025.png


## 5. Principal Investigator Analysis

In [12]:
pip install statsmodels

Note: you may need to restart the kernel to use updated packages.


In [13]:
# pi_name already extracted during data cleaning; verify here
import numpy as np
import plotly.graph_objects as go

pi_total = (
    df.groupby('pi_name')['awd_id']
    .count().reset_index()
    .rename(columns={'awd_id': 'num_awards'})
    .sort_values('num_awards', ascending=False)
    .reset_index(drop=True)
)
pi_total['rank'] = np.arange(len(pi_total))

fig1 = px.bar(
    pi_total, x='pi_name', y='num_awards',
    title='All Principal Investigators by Total Awards (2021–2025)',
    labels={'num_awards': 'Total Awards', 'pi_name': 'Principal Investigator'},
    height=600
)
fig1.update_traces(marker_color='steelblue')

x = pi_total['rank'].values
y_start = pi_total['num_awards'].iloc[0]
y_end = 1
b = np.log(y_start / y_end) / (len(x) * 0.3)  # steeper: 0.3 instead of 1.0
smooth_y = y_start * np.exp(-b * x)

fig1.add_trace(go.Scatter(
    x=pi_total['pi_name'], y=smooth_y,
    mode='lines', name='Trend', line=dict(color='red', width=2)
))

# add equation annotation
equation = f'y = {y_start:.1f} * e^(-{b:.4f}x)'
fig1.add_annotation(
    x=0.98, y=0.95, xref='paper', yref='paper',
    text=equation, showarrow=False,
    font=dict(size=12, color='red'),
    bgcolor='white', bordercolor='red', borderwidth=1
)

fig1.update_layout(xaxis_tickangle=-90, xaxis_tickfont_size=6,
                   plot_bgcolor='white', paper_bgcolor='white')
save_fig(fig1, '14_pi_awards_bar')
fig1.show()

pi_bubble = (
    df.groupby('pi_name').agg(
        total_awards=('awd_id', 'count'),
        total_funding=('amount', 'sum')
    ).reset_index()
    .sort_values('total_awards', ascending=False)
)

fig3 = px.scatter(
    pi_bubble, x='total_awards', y='total_funding',
    size='total_awards', hover_name='pi_name',
    text='pi_name',
    title='All Principal Investigators - Bubble Chart (2021-2025)',
    labels={'total_funding': 'Total Funding ($)', 'total_awards': 'Number of Awards'},
    color='total_funding', color_continuous_scale='Blues', height=600
)
fig3.update_traces(textposition='top center', textfont_size=7)
save_fig(fig3, '16_pi_bubble')
fig3.show()

  Saved → plots/14_pi_awards_bar.png


  Saved → plots/16_pi_bubble.png


In [14]:
# ── Award Duration Distribution ───────────────────────────────────────────────
df['duration_rounded'] = df['duration_years'].round(0)
avg_duration = df['duration_rounded'].dropna()

print(f'Mean award duration:   {avg_duration.mean():.2f} years')
print(f'Median award duration: {avg_duration.median():.2f} years')
print(df['duration_rounded'].value_counts().sort_index())

fig = px.histogram(
    df, x='duration_rounded',
    title='Distribution of NSF Award Durations',
    labels={'duration_rounded': 'Duration (Years)'},
    color_discrete_sequence=['steelblue'],
    nbins=7
)
fig.update_traces(xbins=dict(start=0, end=7, size=1))
fig.add_vline(x=avg_duration.mean(), line_dash='dash', line_color='red',
              annotation_text=f'Mean: {avg_duration.mean():.1f} yrs')
fig.update_layout(bargap=0.1)
save_fig(fig, '17_award_duration_hist')
fig.show()

Mean award duration:   2.91 years
Median award duration: 3.00 years
duration_rounded
0.0     170
1.0     683
2.0    1159
3.0    2382
4.0    1558
5.0     442
6.0       4
Name: count, dtype: int64
  Saved → plots/17_award_duration_hist.png


In [15]:
# ── Institution → PI hierarchical analysis ────────────────────────────────────
inst_pi = (
    df.groupby(['institution', 'pi_name']).agg(
        total_funding=('amount', 'sum'),
        total_awards=('awd_id', 'count')
    ).reset_index()
)

# rank institutions by total funding
inst_totals = (
    inst_pi.groupby('institution')['total_funding']
    .sum().reset_index()
    .rename(columns={'total_funding': 'inst_total'})
    .sort_values('inst_total', ascending=False)
    .reset_index(drop=True)
)
inst_totals['inst_rank'] = inst_totals.index

# merge ranks back and sort: institution rank first, then pi funding within
inst_pi = inst_pi.merge(inst_totals[['institution', 'inst_rank', 'inst_total']], on='institution')
inst_pi = inst_pi.sort_values(
    ['inst_rank', 'total_funding'], ascending=[True, False]
).reset_index(drop=True)

# ── Bar: total funding sorted by school then PI ───────────────────────────────
inst_pi['label'] = inst_pi['institution'] + ' — ' + inst_pi['pi_name']

fig1 = px.bar(
    inst_pi, 
    x='label', y='total_funding',
    color='institution',
    title='PI-Institution Pairs by Total Funding (2021-2025)',
    labels={'total_funding': 'Total Funding ($)', 'label': 'Institution — PI'},
    height=650,
    color_discrete_sequence=pc.qualitative.Light24
)
fig1.update_layout(xaxis_tickangle=-90, xaxis_tickfont_size=7, showlegend=False)
save_fig(fig1, '31_inst_pi_funding')
fig1.show()

# ── Bar: total awards sorted by school then PI ────────────────────────────────
inst_pi_awards = inst_pi.sort_values(
    ['inst_rank', 'total_awards'], ascending=[True, False]
).reset_index(drop=True)

fig2 = px.bar(
    inst_pi_awards,
    x='label', y='total_awards',
    color='institution',
    title='PI-Institution Pairs by Total Awards (2021-2025)',
    labels={'total_awards': 'Number of Awards', 'label': 'Institution — PI'},
    height=650,
    color_discrete_sequence=pc.qualitative.Light24
)
fig2.update_layout(xaxis_tickangle=-90, xaxis_tickfont_size=7, showlegend=False)
save_fig(fig2, '32_inst_pi_awards')
fig2.show()

  Saved → plots/31_inst_pi_funding.png


  Saved → plots/32_inst_pi_awards.png


## 6. Topic Modeling on Award Abstracts
We apply two methods:
- **LDA** — classical bag-of-words probabilistic model
- **BERTopic** — transformer-based with BERT embeddings + UMAP + HDBSCAN

In [16]:
# ── Inspect abstract column ──────────────────────────────────────────────────
print('=== abstract sample ===')
print(df['abstract'].iloc[0][:300])
print(f'abstract non-null: {df["abstract"].notna().sum()}')


=== abstract sample ===
particle and nuclear physics pnp are fundamentally probabilistic due to quantum mechanics both fields rely on complex montecarlo mcbased simulators that use random number sampling to make predictions for nearly all aspects of experimental design and data interpretation in fact most branches of scien
abstract non-null: 6398


In [17]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Stop words: NLTK base + extended coverage + NSF boilerplate ───────────────
stop_words_cs = set(stopwords.words('english'))
stop_words_cs.update([
    'also', 'yet', 'either', 'neither', 'often', 'usually', 'sometimes',
    'always', 'never', 'ever', 'still', 'already', 'just', 'even', 'though',
    'although', 'since', 'unless', 'whether', 'while', 'within', 'without',
    'among', 'along', 'across', 'around', 'toward', 'towards', 'upon',
    'hereby', 'herein', 'thereof', 'therein', 'whereby', 'wherein',
    'something', 'someone', 'somewhere', 'somehow', 'somewhat',
    'anything', 'anyone', 'anywhere', 'nothing', 'nobody', 'nowhere',
    'everything', 'everyone', 'everywhere', 'each', 'every', 'both',
    'several', 'few', 'less', 'least', 'much', 'more', 'most',
    'per', 'via', 'etc', 'ie', 'eg', 'thus', 'hence', 'therefore',
    'moreover', 'furthermore', 'however', 'nevertheless', 'nonetheless',
    'meanwhile', 'thereafter', 'thereby', 'therefrom',
])
stop_words_cs.update([
    'nsf', 'award', 'funded', 'funding', 'university', 'college',
    'investigator', 'researcher', 'student', 'faculty', 'professor',
    'graduate', 'undergraduate', 'phd', 'postdoc',
    'intellectual', 'merit', 'broader', 'impact',
    'mission', 'reflect', 'criterion', 'deem', 'statutory', 'worthy',
    'research', 'study', 'project', 'program', 'support', 'develop',
    'development', 'provide', 'work', 'result', 'include', 'approach',
    'framework', 'tool', 'application', 'technique', 'method', 'propose',
    'proposed', 'present', 'investigate', 'explore', 'examine', 'address',
    'enable', 'allow', 'improve', 'increase', 'reduce', 'important',
    'critical', 'novel', 'innovative', 'effective', 'efficient', 'robust',
    'scalable', 'real', 'large', 'high', 'broad', 'multiple', 'various',
    'different', 'existing', 'current', 'future', 'potential', 'significant',
    'national', 'foundation', 'science', 'scientific', 'team', 'paper',
    'publication', 'dataset', 'experiment', 'make', 'build', 'create',
    'design', 'show', 'demonstrate', 'achieve', 'perform', 'apply',
    'extend', 'combine', 'integrate', 'leverage', 'utilize', 'deploy',
    'implement', 'system', 'model', 'data', 'information', 'knowledge',
    'task', 'process', 'resource', 'environment', 'platform',
    'infrastructure', 'component', 'layer', 'level', 'step', 'stage',
    'aspect', 'feature', 'property', 'factor', 'element', 'type', 'class',
    'category', 'group', 'number', 'case', 'example', 'instance',
    'scenario', 'condition', 'outcome', 'activity', 'opportunity',
    'effort', 'practice', 'enhance', 'identify', 'require', 'lead',
    'share', 'access', 'plan', 'help', 'need', 'understand', 'exist',
    'base', 'diverse', 'complex', 'service', 'material', 'state',
    'structure', 'benefit', 'distribute', 'change', 'evaluate', 'involve',
    'limit', 'range', 'institution', 'course', 'capability', 'energy',
])

# ── Subfield seed terms (expanded via TF-IDF against abstract corpus) ─────────
cs_subfields_seed = {
    'machine_learning':     ['neural', 'deep learning', 'transformer', 'llm', 'generative',
                             'reinforcement', 'supervised', 'unsupervised', 'embedding',
                             'classification', 'backpropagation', 'convolution', 'lstm',
                             'diffusion', 'finetuning', 'inference', 'adversarial'],
    'cybersecurity':        ['security', 'privacy', 'cryptography', 'malware', 'cyber',
                             'encryption', 'authentication', 'intrusion', 'vulnerability',
                             'firewall', 'forensic', 'breach', 'exploit', 'threat',
                             'blockchain', 'trustworthy', 'identity', 'authorization'],
    'networking':           ['network', 'wireless', '5g', 'protocol', 'routing',
                             'bandwidth', 'latency', 'spectrum', 'communication',
                             'internet', 'edge', 'cloud', 'packet', 'topology',
                             'mimo', 'antenna', 'throughput', 'congestion', 'cellular',
                             'wifi', 'iot', 'sensor'],
    'robotics':             ['robot', 'autonomous', 'drone', 'navigation', 'manipulation',
                             'vehicle', 'planning', 'perception', 'control', 'actuator',
                             'locomotion', 'swarm', 'humanoid', 'teleoperation',
                             'haptic', 'grasping', 'mapping', 'slam', 'motion'],
    'hci':                  ['human computer interaction', 'interface', 'usability',
                             'accessibility', 'visualization', 'augmented reality',
                             'virtual reality', 'wearable', 'gesture', 'experience',
                             'cognitive', 'crowdsourcing', 'gamification', 'collaboration'],
    'algorithms':           ['algorithm', 'complexity', 'optimization', 'graph',
                             'combinatorial', 'approximation', 'randomized', 'streaming',
                             'parallel', 'computational', 'geometry', 'sorting',
                             'dynamic programming', 'heuristic', 'polynomial'],
    'systems':              ['operating system', 'compiler', 'kernel', 'gpu', 'architecture',
                             'memory', 'storage', 'cache', 'processor', 'runtime',
                             'virtualization', 'container', 'performance', 'scalability',
                             'fault tolerance', 'scheduling', 'concurrency', 'fpga', 'cpu'],
    'software_engineering': ['software', 'testing', 'debugging', 'verification', 'devops',
                             'specification', 'refactoring', 'maintenance', 'documentation',
                             'agile', 'deployment', 'bug', 'patch', 'repository',
                             'static analysis', 'formal methods'],
    'data_science':         ['database', 'mining', 'analytics', 'clustering', 'bayesian',
                             'query', 'warehouse', 'knowledge graph', 'ontology',
                             'semantic', 'retrieval', 'indexing', 'anomaly', 'pattern',
                             'statistical', 'causal', 'inference', 'fairness'],
    'quantum_computing':    ['quantum', 'qubit', 'entanglement', 'decoherence', 'circuit',
                             'superposition', 'gate', 'error correction', 'annealing',
                             'simulation', 'speedup', 'supremacy'],
    'cyberinfrastructure':  ['cyberinfrastructure', 'hpc', 'supercomputing', 'workflow',
                             'reproducibility', 'cluster', 'grid', 'pipeline',
                             'openscience', 'provenance', 'interoperability', 'metadata',
                             'repository', 'portal', 'gateway', 'middleware'],
    'health_ai':            ['health', 'clinical', 'genomics', 'bioinformatics', 'biomedical',
                             'medical', 'patient', 'diagnosis', 'treatment', 'disease',
                             'imaging', 'drug', 'electronic health record', 'hospital',
                             'monitoring', 'epidemiology', 'precision medicine'],
}

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
tfidf.fit(df['abstract'].dropna())
vocab = tfidf.get_feature_names_out()

def expand_keywords(seeds, vocab):
    expanded = set(seeds)
    for term in vocab:
        if any(s in term for s in seeds):
            expanded.add(term)
    return list(expanded)

cs_subfields = {
    subfield: expand_keywords(seeds, vocab)
    for subfield, seeds in cs_subfields_seed.items()
}
cs_keywords_keep = set(w for kws in cs_subfields.values() for w in kws)

# ── Preprocessing ─────────────────────────────────────────────────────────────
lemmatizer = WordNetLemmatizer()

def preprocess_cs(text):
    if not isinstance(text, str) or len(text) < 20:
        return []
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t, pos='v') for t in tokens]
    tokens = [lemmatizer.lemmatize(t, pos='n') for t in tokens]
    tokens = [t for t in tokens if t not in stop_words_cs and 3 < len(t) < 25]
    return tokens

def cs_keyword_ratio(tokens):
    if not tokens:
        return 0
    return sum(1 for t in tokens if t in cs_keywords_keep) / len(tokens)

# ── Prepare abstract dataframe only ──────────────────────────────────────────
df_text_abstract = df.copy()
df_text_abstract['tokens'] = df_text_abstract['abstract'].apply(preprocess_cs)
df_text_abstract['cs_ratio'] = df_text_abstract['tokens'].apply(cs_keyword_ratio)
df_text_abstract = df_text_abstract[df_text_abstract['tokens'].map(len) >= 5].reset_index(drop=True)
print(f'Abstract rows after filtering: {len(df_text_abstract):,}')


Abstract rows after filtering: 6,398


### 6a. LDA Topic Modeling

In [18]:
def build_gensim_corpus(tokens):
    """Build gensim dictionary and BoW corpus from a list of token lists."""
    dictionary = corpora.Dictionary(tokens)
    dictionary.filter_extremes(no_below=15, no_above=0.4)
    corpus = [dictionary.doc2bow(t) for t in tokens]
    return dictionary, corpus

def find_best_k(tokens, label, k_range=range(5, 55, 5)):
    """Search k from 5 to 50 in steps of 5 for a thorough sweep."""
    dictionary, corpus = build_gensim_corpus(tokens)
    coherence_scores = []
    for k in k_range:
        lda = LdaModel(corpus=corpus, id2word=dictionary,
                       num_topics=k, random_state=42,
                       passes=15, iterations=150,
                       alpha='auto', eta='auto')
        cm = CoherenceModel(model=lda, texts=tokens,
                            dictionary=dictionary, coherence='c_v')
        score = cm.get_coherence()
        coherence_scores.append(score)
        print(f'  [{label}] k={k:2d}: coherence = {score:.4f}')
    best_k = list(k_range)[coherence_scores.index(max(coherence_scores))]
    print(f'\n  [{label}] Best k = {best_k} (coherence = {max(coherence_scores):.4f})\n')
    return best_k, dictionary, corpus, coherence_scores, list(k_range)

print('=== Abstract ===')
best_k_abstract, dictionary_abs, corpus_abs, scores_abs, k_range_abs = find_best_k(
    df_text_abstract['tokens'].tolist(), 'abstract'
)

fig_coh, ax_coh = plt.subplots(figsize=(10, 4))
ax_coh.plot(k_range_abs, scores_abs, marker='o', label='Abstract', color='steelblue')
ax_coh.set_title('LDA Coherence Score by Number of Topics (Abstract)')
ax_coh.set_xlabel('Number of Topics (k)')
ax_coh.set_ylabel('Coherence Score (c_v)')
ax_coh.legend()
plt.tight_layout()
save_fig(fig_coh, '18_lda_coherence')
plt.show()

print(f'Best k — Abstract: {best_k_abstract}')


=== Abstract ===
  [abstract] k= 5: coherence = 0.3978
  [abstract] k=10: coherence = 0.4744
  [abstract] k=15: coherence = 0.4583
  [abstract] k=20: coherence = 0.4804
  [abstract] k=25: coherence = 0.4739
  [abstract] k=30: coherence = 0.4754
  [abstract] k=35: coherence = 0.4578
  [abstract] k=40: coherence = 0.4504
  [abstract] k=45: coherence = 0.4519
  [abstract] k=50: coherence = 0.4487

  [abstract] Best k = 20 (coherence = 0.4804)

  Saved → plots/18_lda_coherence.png
Best k — Abstract: 20


In [19]:
# ── Train final LDA model using best k from coherence sweep ───────────────────
# best_k_abstract is set automatically from the sweep above.
# Override here if you want a fixed k: best_k_abstract = 20
print(f'Training LDA with k={best_k_abstract}')

lda_abstract = LdaModel(
    corpus=corpus_abs, id2word=dictionary_abs,
    num_topics=best_k_abstract, random_state=42,
    passes=40, iterations=400,
    alpha='auto', eta='auto',
    chunksize=2000, minimum_probability=0.01
)
print('── LDA Abstract Topics ──\n')
for idx, topic in lda_abstract.print_topics(num_words=10):
    words = [w.split('*')[1].replace('"','').strip() for w in topic.split('+')]
    print(f'Topic {idx+1:2d}: {" | ".join(words)}')


Training LDA with k=20
── LDA Abstract Topics ──

Topic  1: wireless | sense | spectrum | communication | device | patient | sensor | health | mobile | monitor
Topic  2: security | attack | secure | vulnerability | cybersecurity | threat | detection | defense | authentication | detect
Topic  3: privacy | event | speech | risk | machine | analysis | technology | private | individual | differential
Topic  4: computational | simulation | engineer | device | circuit | technology | physic | physical | storage | flow
Topic  5: decision | fairness | bias | machine | human | time | mechanism | account | fair | algorithm
Topic  6: image | neural | deep | train | representation | object | computer | vision | network | video
Topic  7: intelligence | artificial | agent | train | decisionmaking | healthcare | ensure | machine | aibased | domain
Topic  8: software | code | test | language | verification | developer | formal | automate | reason | analysis
Topic  9: conference | workshop | participant

In [20]:
# ── Assign dominant topic & CS subfield labels ────────────────────────────────
# IMPORTANT: review the topic words printed above and update this dict to match.
# Keys are 0-indexed topic numbers (0 to best_k_abstract-1).
# The labels below match the k=20 best-k output — re-label if k changed.
def get_dominant_topic(bow, model):
    topics = model.get_document_topics(bow)
    return max(topics, key=lambda x: x[1])[0] if topics else -1

abstract_topic_labels = {
    0:  'Networking & Health Sensing',          # wireless | sense | spectrum | patient | health | sensor
    1:  'Cybersecurity & Threat Detection',     # security | attack | vulnerability | threat | detection
    2:  'Privacy & Fairness',                   # privacy | speech | risk | private | differential
    3:  'Scientific Computing & Simulation',    # computational | simulation | engineer | circuit | physics
    4:  'Fairness & Algorithmic Ethics',        # decision | fairness | bias | fair | algorithm
    5:  'Computer Vision & Deep Learning',      # image | neural | deep | object | vision | video
    6:  'AI Agents & Healthcare',               # intelligence | artificial | agent | healthcare | AI-based
    7:  'Software Engineering & Formal Methods',# software | code | verification | formal | automate
    8:  'Academic Community & Outreach',        # conference | workshop | participant | travel | career
    9:  'Biology & Generative AI',              # teacher | generative | biological | molecular | biology
    10: 'Cyberinfrastructure & Open Science',   # cyberinfrastructure | workflow | datasets | open
    11: 'Algorithms, ML & Theory',              # algorithm | graph | theory | optimization | theoretical
    12: 'Social Media & Online Behavior',       # user | social | online | policy | content | behavior
    13: 'Robotics & Autonomous Systems',        # robot | autonomous | vehicle | safety | robotics
    14: 'Quantum & NLP',                        # quantum | language | error | compression | recommender
    15: 'STEM Education & Workforce',           # train | education | school | workforce | skill
    16: 'Networking & Edge Computing',          # network | edge | internet | smart | traffic | testbed
    17: 'HCI & Accessibility',                  # virtual | visualization | interaction | reality | accessibility
    18: 'Resilience & Smart Cities',            # disaster | resilience | city | local | response | mobility
    19: 'Systems & Architecture',               # compute | hardware | memory | architecture | cloud | HPC
}

# Pad any extra topics if best_k > 15
for i in range(len(abstract_topic_labels), best_k_abstract):
    abstract_topic_labels[i] = f'Topic {i+1}'

df_text_abstract['lda_topic'] = [
    get_dominant_topic(bow, lda_abstract) for bow in corpus_abs
]
df_text_abstract['cs_subfield'] = df_text_abstract['lda_topic'].map(abstract_topic_labels)

print('Abstract subfield distribution:')
print(df_text_abstract['cs_subfield'].value_counts())


Abstract subfield distribution:
cs_subfield
STEM Education & Workforce               760
Algorithms, ML & Theory                  668
Systems & Architecture                   596
Resilience & Smart Cities                454
Software Engineering & Formal Methods    428
Cybersecurity & Threat Detection         374
Networking & Health Sensing              367
Academic Community & Outreach            340
Robotics & Autonomous Systems            327
Networking & Edge Computing              291
Computer Vision & Deep Learning          287
Social Media & Online Behavior           279
HCI & Accessibility                      265
Scientific Computing & Simulation        255
Cyberinfrastructure & Open Science       231
Fairness & Algorithmic Ethics            167
Quantum & NLP                            122
Biology & Generative AI                   92
Privacy & Fairness                        69
AI Agents & Healthcare                    26
Name: count, dtype: int64


In [21]:
# ── LDA: Awards & funding per subfield per year ───────────────────────────────
distinct_colors = pc.qualitative.Light24  

subfield_year = (
    df_text_abstract.groupby(['year', 'cs_subfield']).agg(
        num_awards=('awd_id', 'count'),
        total_funding=('amount', 'sum')
    ).reset_index()
)

for metric, ylabel, suffix in [
    ('num_awards',    'Number of Awards', 'awards'),
    ('total_funding', 'Total Funding ($)', 'funding'),
]:
    # ── line chart with distinct colors ──────────────────────────────────────
    fig_line = px.line(
        subfield_year, x='year', y=metric, color='cs_subfield',
        title=f'NSF CSE {ylabel} per CS Subfield per Year',
        labels={metric: ylabel, 'year': 'Year', 'cs_subfield': 'CS Subfield'},
        markers=True, height=550,
        color_discrete_sequence=distinct_colors
    )
    fig_line.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
    save_fig(fig_line, f'19_lda_line_abstract_{suffix}')
    fig_line.show()

    # ── normalized 100% stacked bar for percentage comparison ─────────────────
    total_per_year = subfield_year.groupby('year')[metric].transform('sum')
    subfield_year[f'{metric}_pct'] = subfield_year[metric] / total_per_year * 100

    fig_bar = px.bar(
        subfield_year, x='year', y=f'{metric}_pct', color='cs_subfield',
        title=f'NSF CSE {ylabel} by CS Subfield (% Share per Year)',
        labels={f'{metric}_pct': '% Share', 'year': 'Year', 'cs_subfield': 'CS Subfield'},
        barmode='stack', height=550,
        color_discrete_sequence=distinct_colors
    )
    fig_bar.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
    fig_bar.update_yaxes(range=[0, 100], ticksuffix='%')
    save_fig(fig_bar, f'20_lda_stacked_abstract_{suffix}')
    fig_bar.show()

  Saved → plots/19_lda_line_abstract_awards.png


  Saved → plots/20_lda_stacked_abstract_awards.png


  Saved → plots/19_lda_line_abstract_funding.png


  Saved → plots/20_lda_stacked_abstract_funding.png


In [22]:
# ── Total funding by LDA subfield ────────────────────────────────────────────
topic_funding = (
    df_text_abstract.groupby('cs_subfield')['amount']
    .sum().reset_index()
    .sort_values('amount', ascending=False)
)
fig = px.bar(
    topic_funding, x='amount', y='cs_subfield', orientation='h',
    title='Total NSF Funding by CS Subfield (2021-2025)',
    labels={'amount': 'Total Funding ($)', 'cs_subfield': 'CS Subfield'},
    height=600
)
fig.update_traces(marker_color='steelblue')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
save_fig(fig, '21_lda_funding_abstract')
fig.show()


  Saved → plots/21_lda_funding_abstract.png


In [23]:
# ── LDA topic distribution area chart ────────────────────────────────────────
topic_year = (
    df_text_abstract.groupby(['year', 'cs_subfield'])['awd_id']
    .count().reset_index()
    .rename(columns={'awd_id': 'count'})
)
fig_area = px.area(
    topic_year, x='year', y='count', color='cs_subfield',
    title='LDA Topic Distribution Across Years',
    labels={'count': 'Number of Awards', 'year': 'Year', 'cs_subfield': 'CS Subfield'},
    height=550
)
fig_area.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
save_fig(fig_area, '22_lda_area_abstract')
fig_area.show()


  Saved → plots/22_lda_area_abstract.png


### 6b. BERTopic — Transformer-Based Topic Modeling

In [24]:
# ── Clean text for BERTopic (raw text, not tokens) ────────────────────────────
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

abstracts_list = df_text_abstract['abstract'].apply(clean_text).tolist()
print(f'Abstract docs: {len(abstracts_list):,}')


Abstract docs: 6,398


In [25]:
# ── BERT Embeddings ───────────────────────────────────────────────────────────
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print('Encoding abstracts...')
embeddings_abs = embedding_model.encode(abstracts_list, show_progress_bar=True, batch_size=64)
print(f'Abstract embeddings shape: {embeddings_abs.shape}')


Encoding abstracts...


Batches:   0%|          | 0/100 [00:00<?, ?it/s]

Abstract embeddings shape: (6398, 384)


In [26]:
# ── UMAP + HDBSCAN + BERTopic ─────────────────────────────────────────────────
umap_model    = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                     metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=30, min_samples=10,
                        metric='euclidean', prediction_data=True)
vectorizer    = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))

topic_model_abstract = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    top_n_words=10, verbose=True
)

print('Fitting BERTopic on abstracts...')
topics_abs, probs_abs = topic_model_abstract.fit_transform(abstracts_list, embeddings_abs)
df_text_abstract['bert_topic'] = topics_abs
n_topics_abs   = len(topic_model_abstract.get_topic_info()) - 1
n_outliers_abs = (df_text_abstract['bert_topic'] == -1).sum()
print(f'BERTopic (abstract): {n_topics_abs} topics, {n_outliers_abs:,} outliers '
      f'({n_outliers_abs/len(df_text_abstract)*100:.1f}%)')


2026-06-10 02:04:18,094 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic on abstracts...


2026-06-10 02:04:29,494 - BERTopic - Dimensionality - Completed ✓
2026-06-10 02:04:29,495 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-10 02:04:29,565 - BERTopic - Cluster - Completed ✓
2026-06-10 02:04:29,567 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-10 02:04:30,720 - BERTopic - Representation - Completed ✓


BERTopic (abstract): 61 topics, 1,612 outliers (25.2%)


In [27]:
# ── BERTopic top-word bar charts ──────────────────────────────────────────────
print('── Top Words per BERTopic — Abstract ──\n')
info = topic_model_abstract.get_topic_info()
for _, row in info[info['Topic'] != -1].head(15).iterrows():
    words = [w for w, _ in topic_model_abstract.get_topic(row['Topic'])]
    print(f"T{row['Topic']:3d} ({row['Count']:4,} docs): {' | '.join(words[:8])}")

fig_bc = topic_model_abstract.visualize_barchart(top_n_topics=15, n_words=8)
fig_bc.update_layout(title='BERTopic Top Words — Abstract')
save_fig(fig_bc, '23_bert_barchart_abstract')
fig_bc.show()


── Top Words per BERTopic — Abstract ──

T  0 ( 286 docs): wireless | spectrum | networks | network | communication | communications | radio | 5g
T  1 ( 243 docs): autonomous | safety | systems | learning | control | driving | vehicles | agents
T  2 ( 214 docs): communities | community | resilience | disaster | food | water | emergency | civic
T  3 ( 182 docs): models | medical | learning | clinical | ai | data | project | health
T  4 ( 173 docs): online | social | media | social media | users | information | news | authentication
T  5 ( 165 docs): memory | computing | performance | systems | hardware | parallel | project | design
T  6 ( 149 docs): fairness | fair | algorithmic | data | causal | learning | algorithms | project
T  7 ( 148 docs): network | internet | cloud | performance | dns | applications | project | networks
T  8 ( 141 docs): cs | teachers | computer science | school | computer | computing | students | science
T  9 ( 134 docs): quantum | quantum computing | computing 

In [28]:
# ── BERTopic funding by topic ─────────────────────────────────────────────────
bt_funding = (
    df_text_abstract[df_text_abstract['bert_topic'] != -1]
    .groupby('bert_topic')['amount']
    .sum().reset_index()
    .sort_values('amount', ascending=False)
    .head(15)
)
bt_funding['topic_label'] = bt_funding['bert_topic'].apply(
    lambda t: f"T{t}: " + topic_model_abstract.get_topic(t)[0][0]
)
fig = px.bar(
    bt_funding, x='amount', y='topic_label', orientation='h',
    title='Top 15 BERTopics by Total NSF Funding',
    labels={'amount': 'Total Funding ($)', 'topic_label': 'BERTopic'},
    color='amount', color_continuous_scale='Teal', height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
save_fig(fig, '24_bert_funding_abstract')
fig.show()


  Saved → plots/24_bert_funding_abstract.png


In [29]:
# ── BERTopic topics over time ─────────────────────────────────────────────────
top_topics = (
    df_text_abstract[df_text_abstract['bert_topic'] != -1]['bert_topic']
    .value_counts().head(10).index.tolist()
)
topic_year = (
    df_text_abstract[df_text_abstract['bert_topic'].isin(top_topics)]
    .groupby(['year', 'bert_topic'])['awd_id']
    .count().reset_index()
    .rename(columns={'awd_id': 'count'})
)
topic_year['topic_label'] = topic_year['bert_topic'].apply(
    lambda t: f"T{t}: " + topic_model_abstract.get_topic(t)[0][0]
)
fig_line = px.line(
    topic_year, x='year', y='count', color='topic_label',
    title='BERTopic Top 10 Topics Over Time',
    labels={'count': 'Number of Awards', 'year': 'Year', 'topic_label': 'Topic'},
    markers=True, height=550
)
fig_line.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
save_fig(fig_line, '25_bert_topics_over_time_abstract')
fig_line.show()

fig_bar = px.bar(
    topic_year, x='year', y='count', color='topic_label',
    title='BERTopic Topic Distribution per Year',
    labels={'count': 'Number of Awards', 'year': 'Year', 'topic_label': 'Topic'},
    barmode='stack', height=550
)
fig_bar.update_xaxes(tickvals=[2021, 2022, 2023, 2024, 2025])
save_fig(fig_bar, '26_bert_dist_per_year_abstract')
fig_bar.show()


  Saved → plots/25_bert_topics_over_time_abstract.png


  Saved → plots/26_bert_dist_per_year_abstract.png


## 7. LDA vs. BERTopic Comparison

In [30]:
# ── Heatmap: LDA vs BERTopic assignment overlap ───────────────────────────────
overlap = df_text_abstract[df_text_abstract['bert_topic'] != -1].copy()
overlap['lda_label']  = overlap['cs_subfield']
overlap['bert_label'] = 'BERT_' + overlap['bert_topic'].astype(str)

crosstab = pd.crosstab(overlap['lda_label'], overlap['bert_label']).iloc[:, :20]

fig_ht, ax_ht = plt.subplots(figsize=(18, 8))
sns.heatmap(crosstab, cmap='YlOrRd', linewidths=0.2, annot=False, ax=ax_ht)
ax_ht.set_title('LDA vs BERTopic Assignment Overlap', fontsize=14, fontweight='bold')
ax_ht.set_xlabel('BERTopic')
ax_ht.set_ylabel('LDA CS Subfield')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
save_fig(fig_ht, '27_lda_bert_heatmap_abstract')
plt.show()


  Saved → plots/27_lda_bert_heatmap_abstract.png


In [31]:
# ── Subfield distribution bar chart ──────────────────────────────────────────
abs_dist = df_text_abstract['cs_subfield'].value_counts().reset_index()
abs_dist.columns = ['subfield', 'count']
abs_dist['pct'] = abs_dist['count'] / abs_dist['count'].sum() * 100

fig_dist = px.bar(
    abs_dist, x='pct', y='subfield', orientation='h',
    title='NSF CSE Abstract Subfield Distribution (% of Awards)',
    labels={'pct': '% of Awards', 'subfield': 'CS Subfield'},
    height=600
)
fig_dist.update_traces(marker_color='steelblue')
fig_dist.update_layout(yaxis={'categoryorder': 'total ascending'})
save_fig(fig_dist, '28_subfield_distribution')
fig_dist.show()


  Saved → plots/28_subfield_distribution.png


In [32]:
# ── Top growing vs declining subfields ────────────────────────────────────────
subfield_year = (
    df_text_abstract.groupby(['year', 'cs_subfield']).agg(
        num_awards=('awd_id', 'count'),
        total_funding=('amount', 'sum')
    ).reset_index()
)
for metric, ylabel, slug in [
    ('num_awards',    '% Change in Number of Awards', 'awards'),
    ('total_funding', '% Change in Total Funding',    'funding'),
]:
    pivot = subfield_year.pivot(index='cs_subfield', columns='year', values=metric).fillna(0)
    pivot['pct_change'] = (
        (pivot[2025] - pivot[2021]) / pivot[2021].replace(0, np.nan) * 100
    ).round(1)
    pivot = pivot.dropna(subset=['pct_change']).sort_values('pct_change', ascending=False)
    colors = ['green' if x >= 0 else 'red' for x in pivot['pct_change']]

    fig_g, ax_g = plt.subplots(figsize=(12, 7))
    ax_g.barh(pivot.index, pivot['pct_change'], color=colors)
    ax_g.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
    ax_g.set_title(f'CS Subfield Growth 2021-2025 ({ylabel})',
                   fontsize=13, fontweight='bold')
    ax_g.set_xlabel(ylabel)
    plt.tight_layout()
    save_fig(fig_g, f'29_growth_abstract_{slug}')
    plt.show()


  Saved → plots/29_growth_abstract_awards.png
  Saved → plots/29_growth_abstract_funding.png


In [33]:
# ── Heatmap: subfield x year — Total Funding ($M) ────────────────────────────
subfield_year = (
    df_text_abstract.groupby(['year', 'cs_subfield'])['amount']
    .sum().reset_index()
)
pivot = subfield_year.pivot(
    index='cs_subfield', columns='year', values='amount'
).fillna(0) / 1e6

fig_hm, ax_hm = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.3, cbar_kws={'label': 'Funding ($M)'}, ax=ax_hm)
ax_hm.set_title('NSF CSE Funding by CS Subfield and Year ($M)',
                fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig_hm, '30_subfield_year_heatmap_abstract')
plt.show()


  Saved → plots/30_subfield_year_heatmap_abstract.png


In [34]:
# ── Heatmap: CS Subfield x Year — Number of Awards ───────────────────────────
subfield_year = (
    df_text_abstract.groupby(['year', 'cs_subfield'])['awd_id']
    .count().reset_index()
    .rename(columns={'awd_id': 'num_awards'})
)
pivot = subfield_year.pivot(
    index='cs_subfield', columns='year', values='num_awards'
).fillna(0).astype(int)
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

fig = px.imshow(
    pivot,
    text_auto=True,
    color_continuous_scale='Blues',
    aspect='auto',
    title='NSF CSE Number of Awards per CS Subfield per Year',
    labels={'x': 'Year', 'y': 'CS Subfield', 'color': 'Number of Awards'},
)
fig.update_layout(
    height=100 + 60 * len(pivot),
    xaxis=dict(tickvals=list(pivot.columns)),
    yaxis=dict(tickfont=dict(size=11)),
    coloraxis_colorbar=dict(title='Awards'),
)
fig.update_traces(textfont=dict(size=11))
save_fig(fig, '30_subfield_awards_heatmap_abstract')
fig.show()


  Saved → plots/30_subfield_awards_heatmap_abstract.png


## 8. Save Results

In [35]:
# ── Save enriched CSV dataset ─────────────────────────────────────────────────
out_cols = ['awd_id', 'awd_titl_txt', 'year', 'amount', 'duration_years',
            'institution', 'state', 'div_abbr', 'pi_name',
            'lda_topic', 'cs_subfield', 'bert_topic']

df_text_abstract[[c for c in out_cols if c in df_text_abstract.columns]].to_csv(
    'nsf_awards_abstract_topics.csv', index=False
)
print('Saved: nsf_awards_abstract_topics.csv')


Saved: nsf_awards_abstract_topics.csv


In [36]:
# ── Save LDA & BERTopic models ────────────────────────────────────────────────
lda_abstract.save('lda_abstract_model')
print('LDA model saved')

topic_model_abstract.save(
    'bertopic_abstract_model', serialization='safetensors', save_ctfidf=True
)
print('BERTopic model saved')


LDA model saved
BERTopic model saved


## 9. Export All Plots to PDF

In [37]:
from PIL import Image
from pypdf import PdfWriter, PdfReader
import io

print(f'Plots registered: {len(SAVED_PLOTS)}')
for p in SAVED_PLOTS:
    exists = os.path.exists(p)
    print(f'  {p}  ← {"OK" if exists else "MISSING"}')

Plots registered: 37
  plots/01_total_funding_per_year.png  ← OK
  plots/02_cfda_funding_trends.png  ← OK
  plots/04_cfda_treemap.png  ← OK
  plots/05_div_funding_per_year.png  ← OK
  plots/06_div_awards_per_year.png  ← OK
  plots/07_div_total_funding.png  ← OK
  plots/08_div_total_awards.png  ← OK
  plots/10_inst_total_bar.png  ← OK
  plots/11_inst_treemap.png  ← OK
  plots/12_inst_bubble.png  ← OK
  plots/13_state_map_2021.png  ← OK
  plots/13_state_map_2022.png  ← OK
  plots/13_state_map_2023.png  ← OK
  plots/13_state_map_2024.png  ← OK
  plots/13_state_map_2025.png  ← OK
  plots/14_pi_awards_bar.png  ← OK
  plots/16_pi_bubble.png  ← OK
  plots/17_award_duration_hist.png  ← OK
  plots/31_inst_pi_funding.png  ← OK
  plots/32_inst_pi_awards.png  ← OK
  plots/18_lda_coherence.png  ← OK
  plots/19_lda_line_abstract_awards.png  ← OK
  plots/20_lda_stacked_abstract_awards.png  ← OK
  plots/19_lda_line_abstract_funding.png  ← OK
  plots/20_lda_stacked_abstract_funding.png  ← OK
  plots/21

In [38]:
# ── Convert each PNG → single-page PDF, then merge ────────────────────────────
# This approach is robust: no LaTeX, no nbconvert, no browser — pure Python.

PDF_OUTPUT = 'NSF_Funding_Analysis_plots.pdf'
writer     = PdfWriter()
skipped    = []

for png_path in SAVED_PLOTS:
    if not os.path.exists(png_path):
        skipped.append(png_path)
        continue
    try:
        img = Image.open(png_path).convert('RGB')
        page_buf = io.BytesIO()
        # A4 landscape at 150 dpi
        img.save(page_buf, format='PDF', resolution=150)
        page_buf.seek(0)
        reader = PdfReader(page_buf)
        for page in reader.pages:
            writer.add_page(page)
    except Exception as e:
        print(f'  WARNING: could not add {png_path}: {e}')
        skipped.append(png_path)

with open(PDF_OUTPUT, 'wb') as f:
    writer.write(f)

size_mb = os.path.getsize(PDF_OUTPUT) / 1e6
print(f'\nPDF saved → {PDF_OUTPUT}  ({size_mb:.1f} MB, {len(writer.pages)} pages)')
if skipped:
    print(f'Skipped {len(skipped)} missing files: {skipped}')


PDF saved → NSF_Funding_Analysis_plots.pdf  (6.2 MB, 37 pages)
